# **Training GLiNER model with MD-related simulation's description** ⚙️

[GLiNER2](https://github.com/fastino-ai/GLiNER2) unifies Named Entity Recognition, Text Classification, Structured Data Extraction, and Relation Extraction into a single 205M parameter model. It provides efficient CPU-based inference without requiring complex pipelines or external API dependencies.

- See Documentation: https://urchade.github.io/GLiNER/
- See Preprint: https://arxiv.org/abs/2311.08526


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from dotenv import load_dotenv
from gliner2 import GLiNER2
from gliner2.training.data import InputExample
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig
from mdner_llm.utils.load_selected_annotation_paths import get_selected_annotation_paths

from mdner_llm.models.entities_with_positions import ListOfEntitiesPositions
from mdner_llm.utils.evaluate_gliner2 import compare_entities
from mdner_llm.utils.visualize_annotations import (
    convert_ner_response_to_entities,
    visualize_llm_annotation,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load models


In [2]:
# Load model with 205M parameters (base version)
base_extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
print(base_extractor)

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
GLiNER2(
  (encoder): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128011, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
    

In [3]:
# Load model with 340M parameters (large version)
# large_extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")
# print(large_extractor)

In [4]:
# Load model GLiNER XL 1B via API
# Note that it need api key from https://pioneer.ai/settings/api-keys
# PIONEER_API_KEY in env file
load_dotenv()
xl_extractor = GLiNER2.from_api()
print(xl_extractor)

## Define entities classes


In [5]:
# Class of entities
entities_class_with_description = {
    "MOL": "Molecule or chemical compound involved in the simulation",
    "SOFTNAME": "Molecular dynamics software used for the simulation",
    "SOFTVERS": "Version of the molecular dynamics software",
    "TEMP": "Simulation temperature, typically expressed in Kelvin or Celcius",
    "FFM": "Force field model used to describe interatomic interactions",
    "STIME": "Total simulation time or duration",
}

## Prepare training data

In [19]:
# Load annotations
selected_annotation_paths = get_selected_annotation_paths(
    annotation_dir=Path("../annotations/v3"),
    selected_files_txt=Path("../results/50_selected_files_20260226_102820.txt"),
)
print(f"First 3 selected annotation paths: {selected_annotation_paths[:3]}")


2026-03-02 12:05:45 | INFO     | Scanning annotation directory: ../annotations/v3
2026-03-02 12:05:45 | INFO     | Total annotation files found: 360
2026-03-02 12:05:45 | INFO     | Number of file paths listed in ../results/50_selected_files_20260226_102820.txt: 50
2026-03-02 12:05:45 | INFO     | Number of matching selected annotation files: 50
First 3 selected annotation paths: [PosixPath('../annotations/v3/zenodo_3248612.json'), PosixPath('../annotations/v3/zenodo_838635.json'), PosixPath('../annotations/v3/zenodo_14594.json')]


In [ ]:
# Create dataset for training

In [ ]:
# Split into train/validation

## Configure training

In [ ]:
model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
config = TrainingConfig(
    output_dir="../results/ner_model",
    experiment_name="gliner2_finetuned_50_descriptions",

    # Training steps
    num_epochs=10,
    max_steps=-1,

    # Batch size
    batch_size=32,
    eval_batch_size=64,
    gradient_accumulation_steps=1,

    # Learning rates
    encoder_lr=1e-5,
    task_lr=5e-4,

    # Early stopping
    early_stopping=False,
    early_stopping_patience=3,
    early_stopping_threshold=0.0,

    # Mixed precision
    fp16=True,

    # Checkpointing & Evaluation (saves when evaluating)
    eval_strategy="epoch",  # "epoch", "steps", or "no"
    metric_for_best="eval_loss",
    save_best=True,

    # Logging
    logging_steps=50,
)

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


## Train

In [ ]:
trainer = GLiNER2Trainer(model, config)
trainer.train(train_data=train_data, val_data=val_data)

## Load best model

In [ ]:
model = GLiNER2.from_pretrained("./gliner2_finetuned_50_descriptions/best")
model